In [1]:
import pandas as pd

#데이터 전처리를 위한 전처리

# 파일 경로
file_path = "/content/2024년 _신입생 충원 현황_학과별자료.csv"

# UTF-8-SIG 인코딩으로 CSV 파일 불러오기
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 컬럼 특수문자, 공백 제거
df['학교'] = df['학교'].str.strip()
df['학교'] = df['학교'].str.replace(' ', '', regex=False)
df['학과'] = df['학과'].str.replace('ㆍ', '·').str.replace('・', '·').str.replace('․', '·').str.replace(' ', '', regex=False).str.replace('-', '', regex=False)

# 주요 컬럼 이름 파악
department_col = [col for col in df.columns if '학과' in col or '전공' in col][0]

# 학과명에서 7자리 숫자 코드 제거
df[department_col] = df[department_col].str.replace(r"\(\d{7}\)", "", regex=True).str.strip()

# 학교명에서 '분교', '_제n캠퍼스' 제거
df["학교"] = df["학교"].str.replace(r"_분교", "", regex=True)
df["학교"] = df["학교"].str.replace(r"_제\d+캠퍼스", "", regex=True)
df.columns = df.columns.str.strip()

# 학교명에서 '국립'으로 시작하지만 '국립경국대학교'는 제외하고 '국립' 제거
mask = df["학교"].str.startswith('국립') & (df["학교"] != '국립경국대학교')
df.loc[mask, "학교"] = df.loc[mask, "학교"].str.replace('^국립', '', regex=True).str.strip()

# 전처리된 데이터를 새로운 CSV 파일로 저장
output_path = "전처리된_신입생_충원_현황(2024).csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')

In [2]:
import pandas as pd

#대계열 분류

# 파일 불러오기
df_main = pd.read_csv("전처리된_신입생_충원_현황(2024).csv", encoding="utf-8")
df_ref = pd.read_csv("/content/2024년 주,야간 정리.csv", encoding="utf-8")

# 컬럼명 특수문자, 공백 제거
df_main.columns = df_main.columns.str.strip()
df_ref.columns = df_ref.columns.str.strip()
df_ref['학과명'] = df_ref['학과명'].str.replace('ㆍ', '·').str.replace('・', '·').str.replace('․', '·').str.replace(' ', '', regex=False).str.replace('-', '', regex=False)

# 중복 제거: 학교명 + 학과명 조합으로 하나만 남기기
df_ref_unique = df_ref.drop_duplicates(subset=["학교명", "학과명"])

# 병합
merged = pd.merge(
    df_main,
    df_ref_unique[["학교명", "학과명", "대계열"]],
    left_on=["학교", "학과"],
    right_on=["학교명", "학과명"],
    how="left"
)

# 중복 컬럼 제거
merged.drop(columns=["학교명", "학과명"], inplace=True)

# 저장
merged.to_csv("신입생_충원_현황(2024)대계열.csv", index=False, encoding="utf-8-sig")

In [13]:
import pandas as pd
import glob
import os

#

# 처리할 CSV 파일들 (파일 이름에 "대계열 0제거"가 들어가는 경우)
file_paths = glob.glob("/content/신입생_충원_현황(20*)대계열 0제거.csv")

# 합산할 컬럼과 그룹 기준 정의
columns_to_sum = ['입학정원(A)', '정원내 모집인원(B)', '정원내 지원자(C)', '정원내 입학자(D)']
group_keys = ['지역', '학교', '대계열']

# 각 파일 반복 처리
for file_path in file_paths:
    # 파일 불러오기
    df = pd.read_csv(file_path, encoding="utf-8")
    df.columns = df.columns.str.strip()

    # 숫자형 변환: 쉼표 제거 후 float 변환
    for col in columns_to_sum:
        df[col] = df[col].astype(str).str.replace(",", "").astype(float)

    # 그룹화 및 합계
    df_grouped = df.groupby(group_keys, as_index=False)[columns_to_sum].sum()

    # 저장 파일 이름 구성
    file_name = os.path.basename(file_path)
    new_name = file_name.replace("대계열 0제거", "통합_수정")
    save_path = os.path.join("/content/", new_name)

    # 저장
    df_grouped.to_csv(save_path, index=False, encoding="utf-8-sig")


In [1]:
import pandas as pd
import glob
import os

# 처리할 파일들 (대계열 0제거 파일들 대상)
file_paths = glob.glob("/content/*대계열 0제거.csv")

# 기본 그룹 키
group_keys = ['지역', '학교', '대계열']

# 합산할 수치 컬럼
numeric_cols = ['입학정원(A)', '정원내 모집인원(B)', '정원내 지원자(C)', '정원내 입학자(D)']

for file_path in file_paths:
    # 파일 불러오기
    df = pd.read_csv(file_path, encoding="utf-8")
    df.columns = df.columns.str.strip()

    # 숫자형 컬럼: 쉼표 제거 후 float 변환
    for col in numeric_cols:
        df[col] = df[col].astype(str).str.replace(",", "").astype(float)

    # 모든 컬럼 정리: 그룹 키 + 합산 컬럼 외 나머지
    other_cols = [col for col in df.columns if col not in group_keys + numeric_cols]

    # agg 딕셔너리 구성
    agg_dict = {col: 'sum' for col in numeric_cols}
    agg_dict.update({col: 'first' for col in other_cols})

    # 그룹화 및 집계
    df_grouped = df.groupby(group_keys, as_index=False).agg(agg_dict)

    # 충원율 및 경쟁률 계산
    df_grouped['충원율(%)'] = (df_grouped['정원내 입학자(D)'] / df_grouped['정원내 모집인원(B)']) * 100
    df_grouped['경쟁률'] = df_grouped['정원내 지원자(C)'] / df_grouped['정원내 모집인원(B)']

    # 반올림
    df_grouped['충원율(%)'] = df_grouped['충원율(%)'].round(2)
    df_grouped['경쟁률'] = df_grouped['경쟁률'].round(2)

    # 파일 저장
    file_name = os.path.basename(file_path)
    save_name = file_name.replace("대계열 0제거", "통합완성")
    save_path = os.path.join("/content/", save_name)

    df_grouped.to_csv(save_path, index=False, encoding="utf-8-sig")



In [5]:
import pandas as pd
import glob
import os

# 통합완성 파일 경로 가져오기
file_paths = glob.glob("/content/*통합완성.csv")

for file_path in file_paths:
    # 파일 불러오기
    df = pd.read_csv(file_path, encoding="utf-8")
    df.columns = df.columns.str.strip()

    # 충원율 기준으로 그룹 통계 계산 (지역, 설립구분, 대계열 기준)
    grouped_stats = df.groupby(['지역', '설립구분', '대계열'])['충원율(%)'].agg(['mean', 'std']).reset_index()
    grouped_stats.rename(columns={'mean': '충원율_평균', 'std': '충원율_표준편차'}, inplace=True)

    # 소수점 4자리 반올림
    grouped_stats['충원율_평균'] = grouped_stats['충원율_평균'].round(4)
    grouped_stats['충원율_표준편차'] = grouped_stats['충원율_표준편차'].round(4)

    # 원본 데이터에 병합
    df = pd.merge(df, grouped_stats, how='left', on=['지역', '설립구분', '대계열'])

    # 안정성 비율 = 표준편차 / 평균
    df['충원안정성비율'] = (df['충원율_표준편차'] / df['충원율_평균']).round(4)

    # NaN 처리 (학과가 하나뿐인 그룹은 표준편차와 안정성비율이 계산 불가 → 0으로)
    df[['충원율_표준편차', '충원안정성비율']] = df[['충원율_표준편차', '충원안정성비율']].fillna(0)

    # 결과 저장
    file_name = os.path.basename(file_path)
    save_name = file_name.replace("통합완성", "통합완성_안정성")
    save_path = os.path.join("/content/", save_name)

    df.to_csv(save_path, index=False, encoding="utf-8-sig")
